# Pearson approximation loss in EXFOR→ENDF Legendre sampling

Compare two parquets produced by `scripts/exfor_to_endf_sampling_v2.py`:

- `legendre_coefficients_all_samples.parquet` — the Cholesky-regenerated samples that the pipeline writes by default in two-pass mode. They are multivariate Gaussian by construction (drawn from `cov_combined = corr_kw × outer(std_perbin, std_perbin)`).
- `legendre_coefficients_full_correlations.parquet` — the raw multi-bin Monte Carlo samples from the kernel-weight pass, written when `SAVE_FULL_CORRELATION_SAMPLES = True`. They carry the full joint distribution from shared-perturbation coupling.

The notebook quantifies how much the Gaussian step distorts the joint distribution of the Legendre coefficients.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

In [ ]:
# Edit RUN_DIR to point at the rerun output directory.
RUN_DIR = Path('/share_snc/snc/JuanMonleon/ENDF_samples/new_test_41')

PARQUETS = {
    'gaussian':          RUN_DIR / 'legendre_coefficients_all_samples.parquet',
    'full_correlations': RUN_DIR / 'legendre_coefficients_full_correlations.parquet',
}

# Match MAX_SAMPLE_ORDER from the pipeline run
MAX_ORDER = 3

COLORS = {
    'gaussian':          '#0173B2',  # blue
    'full_correlations': '#DE8F05',  # orange
}
LABELS = {
    'gaussian':          'Gaussian (Cholesky)',
    'full_correlations': 'Full correlations (raw MC)',
}

for key, path in PARQUETS.items():
    print(f'  {key:18s}  {"OK" if path.exists() else "MISSING":7s}  {path}')

## Loader

Reshape each parquet into a sample matrix `X` of shape `(N_samples, N_params)` with parameter ordering `[a_1(E_1), a_2(E_1), ..., a_L(E_1), a_1(E_2), ...]`. This matches the layout used inside `compute_covariance_from_samples` in `scripts/exfor_utils.py`.

In [ ]:
def load_sample_matrix(parquet_path, max_order):
    """Return (X, energies, param_labels) for a Legendre coefficient parquet.

    X has shape (n_samples, n_energies * max_order). The nominal row
    (sample_idx == 0) is dropped.
    """
    df = pd.read_parquet(parquet_path)
    df = df[df['sample_idx'] > 0].copy()

    energies = (
        df[['energy_index', 'energy_mev']]
        .drop_duplicates()
        .sort_values('energy_index')
        .reset_index(drop=True)
    )
    energy_indices = energies['energy_index'].to_numpy()
    energy_mev = energies['energy_mev'].to_numpy()
    n_energies = len(energy_indices)

    coeff_cols = [f'a_{l + 1}' for l in range(max_order)]
    sample_ids = np.sort(df['sample_idx'].unique())
    n_samples = len(sample_ids)

    X = np.zeros((n_samples, n_energies * max_order))
    e_pos = {e: k for k, e in enumerate(energy_indices)}
    s_pos = {s: i for i, s in enumerate(sample_ids)}
    for row in df.itertuples(index=False):
        i = s_pos[row.sample_idx]
        k = e_pos[row.energy_index]
        start = k * max_order
        for l, col in enumerate(coeff_cols):
            X[i, start + l] = getattr(row, col)

    param_labels = [(int(e), l + 1) for e in energy_indices for l in range(max_order)]
    return X, energy_mev, param_labels


datasets = {}
for key, path in PARQUETS.items():
    if not path.exists():
        print(f'  skipping {key}: {path} not found')
        continue
    X, energy_mev, param_labels = load_sample_matrix(path, MAX_ORDER)
    datasets[key] = {'X': X, 'energies': energy_mev, 'labels': param_labels}
    print(f'  {key:18s}  X.shape = {X.shape}  ({len(energy_mev)} energies x {MAX_ORDER} orders)')

if 'full_correlations' not in datasets:
    print('\nWARNING: legendre_coefficients_full_correlations.parquet not found.')
    print('Re-run scripts/exfor_to_endf_sampling_v2.py with SAVE_FULL_CORRELATION_SAMPLES = True.')

## Section 1 — Marginal diagnostics

Per-parameter skewness, excess kurtosis, and a normality test. The Cholesky set should look Gaussian (skew ≈ 0, excess kurt ≈ 0, p-values uniformly distributed). The full-correlations set shows whatever non-Gaussianity the multi-bin MC actually produced.

In [ ]:
def marginal_stats(X):
    skew = stats.skew(X, axis=0, bias=False)
    exkurt = stats.kurtosis(X, axis=0, bias=False, fisher=True)
    # D'Agostino-Pearson normality test (skew + kurtosis)
    pvals = np.full(X.shape[1], np.nan)
    for i in range(X.shape[1]):
        col = X[:, i]
        if np.std(col) == 0:
            continue
        try:
            pvals[i] = stats.normaltest(col).pvalue
        except Exception:
            pass
    return skew, exkurt, pvals


stats_by_set = {key: marginal_stats(d['X']) for key, d in datasets.items()}

for key, (skew, exkurt, pvals) in stats_by_set.items():
    print(f'{LABELS[key]}:')
    print(f'  |skew|     median = {np.nanmedian(np.abs(skew)):.3f}, max = {np.nanmax(np.abs(skew)):.3f}')
    print(f'  |excess k| median = {np.nanmedian(np.abs(exkurt)):.3f}, max = {np.nanmax(np.abs(exkurt)):.3f}')
    print(f'  D\'Agostino p < 0.05 in {np.sum(pvals < 0.05)}/{np.sum(np.isfinite(pvals))} parameters')
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for key, (skew, exkurt, _) in stats_by_set.items():
    axes[0].hist(skew[np.isfinite(skew)], bins=30, alpha=0.55,
                 color=COLORS[key], label=LABELS[key])
    axes[1].hist(exkurt[np.isfinite(exkurt)], bins=30, alpha=0.55,
                 color=COLORS[key], label=LABELS[key])

axes[0].axvline(0, color='k', lw=0.7)
axes[0].set_xlabel('Sample skewness')
axes[0].set_ylabel('# parameters')
axes[0].set_title('Marginal skewness')
axes[0].legend()

axes[1].axvline(0, color='k', lw=0.7)
axes[1].set_xlabel('Sample excess kurtosis')
axes[1].set_ylabel('# parameters')
axes[1].set_title('Marginal excess kurtosis')
axes[1].legend()

fig.tight_layout()

In [ ]:
# Q-Q plots for a handful of representative parameters
ref_key = 'full_correlations' if 'full_correlations' in datasets else next(iter(datasets))
ref = datasets[ref_key]
n_energies = len(ref['energies'])

# Pick: a_1 at low/mid/high energy, plus a_2 and a_3 at the highest-variance bin
var_a1 = ref['X'][:, 0::MAX_ORDER].var(axis=0)
high_var_bin = int(np.argmax(var_a1))
selections = [
    (0,                                 'a_1, lowest E'),
    (n_energies // 2 * MAX_ORDER + 0,   'a_1, mid E'),
    ((n_energies - 1) * MAX_ORDER + 0,  'a_1, highest E'),
    (high_var_bin * MAX_ORDER + 0,      f'a_1, max-var bin (E={ref["energies"][high_var_bin]:.2f} MeV)'),
    (high_var_bin * MAX_ORDER + 1,      f'a_2, max-var bin'),
    (high_var_bin * MAX_ORDER + 2,      f'a_3, max-var bin') if MAX_ORDER >= 3 else None,
]
selections = [s for s in selections if s is not None]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (idx, title) in zip(axes.flat, selections):
    for key, d in datasets.items():
        col = d['X'][:, idx]
        if np.std(col) == 0:
            continue
        z = (col - col.mean()) / col.std(ddof=1)
        z_sorted = np.sort(z)
        n = len(z_sorted)
        theo = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)
        ax.plot(theo, z_sorted, '.', ms=3, color=COLORS[key], label=LABELS[key])
    lims = [-3.5, 3.5]
    ax.plot(lims, lims, 'k-', lw=0.7)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Theoretical N(0,1)')
    ax.set_ylabel('Sample quantile')
axes[0, 0].legend(loc='upper left', fontsize=8)
fig.suptitle('Q-Q plots: Cholesky vs raw MC samples')
fig.tight_layout()

## Section 2 — Pairwise dependence (correlation matrices)

Side-by-side Pearson correlation matrices, plus their entrywise difference. The difference panel shows where the Gaussian step shifted correlations away from the raw MC structure.

In [ ]:
def safe_corr(X):
    return np.corrcoef(X, rowvar=False)


corrs = {key: safe_corr(d['X']) for key, d in datasets.items()}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
im0 = axes[0].imshow(corrs.get('gaussian', np.zeros((1, 1))),
                     cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_title(LABELS['gaussian'])
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(corrs.get('full_correlations', np.zeros((1, 1))),
                     cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_title(LABELS['full_correlations'])
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

if 'gaussian' in corrs and 'full_correlations' in corrs:
    diff = corrs['full_correlations'] - corrs['gaussian']
    vmax = np.nanmax(np.abs(diff))
    im2 = axes[2].imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    axes[2].set_title(f'Full − Gaussian (max |d|={vmax:.3f})')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

for ax in axes:
    ax.set_xlabel('Parameter index')
    ax.set_ylabel('Parameter index')
fig.tight_layout()

In [ ]:
# Pairwise scatter for a few representative parameter pairs
n_energies = len(ref['energies'])
max_var_bin = int(np.argmax(ref['X'][:, 0::MAX_ORDER].var(axis=0)))

pair_specs = [
    ((max_var_bin * MAX_ORDER + 0, max_var_bin * MAX_ORDER + 1),                 'a1 vs a2 (same bin, max var)'),
    ((max_var_bin * MAX_ORDER + 0, min(max_var_bin + 1, n_energies - 1) * MAX_ORDER + 0), 'a1 vs a1 (adjacent bin)'),
    ((0,                            (n_energies - 1) * MAX_ORDER + 0),           'a1 vs a1 (low E vs high E)'),
]

fig, axes = plt.subplots(len(pair_specs), len(datasets),
                         figsize=(4 * len(datasets), 3.5 * len(pair_specs)))
if len(datasets) == 1:
    axes = axes.reshape(-1, 1)

for row, ((i, j), label) in enumerate(pair_specs):
    for col, (key, d) in enumerate(datasets.items()):
        ax = axes[row, col]
        x = d['X'][:, i]
        y = d['X'][:, j]
        rho = np.corrcoef(x, y)[0, 1]
        ax.plot(x, y, '.', ms=3, alpha=0.5, color=COLORS[key])
        ax.set_title(f'{LABELS[key]}\n{label}\nρ = {rho:+.3f}', fontsize=9)
        ax.set_xlabel(f'param {i}')
        ax.set_ylabel(f'param {j}')
fig.tight_layout()

In [ ]:
# Mutual information from samples vs Gaussian-equivalent MI = -0.5 log(1 - rho^2)
def mi_2d_histogram(x, y, bins=20):
    H, xe, ye = np.histogram2d(x, y, bins=bins)
    pxy = H / H.sum()
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    nz = pxy > 0
    return float(np.sum(pxy[nz] * np.log(pxy[nz] / (px @ py)[nz])))


rows = []
for (i, j), label in pair_specs:
    for key, d in datasets.items():
        x = d['X'][:, i]
        y = d['X'][:, j]
        if np.std(x) == 0 or np.std(y) == 0:
            continue
        rho = np.corrcoef(x, y)[0, 1]
        mi_emp = mi_2d_histogram(x, y, bins=20)
        mi_gauss = -0.5 * np.log(max(1 - rho ** 2, 1e-12))
        rows.append({
            'pair':       label,
            'sample_set': LABELS[key],
            'rho':        rho,
            'MI_empir':   mi_emp,
            'MI_gauss':   mi_gauss,
            'gap':        mi_emp - mi_gauss,
        })

mi_df = pd.DataFrame(rows)
mi_df

## Section 3 — Variance comparison

Per-parameter relative standard deviation as a function of energy, one panel per Legendre order. Shows whether the per-bin variance baked into the Cholesky path differs meaningfully from the variance the raw multi-bin MC produces.

In [ ]:
fig, axes = plt.subplots(1, MAX_ORDER, figsize=(4.5 * MAX_ORDER, 4), sharey=False)
if MAX_ORDER == 1:
    axes = [axes]

for l in range(MAX_ORDER):
    ax = axes[l]
    for key, d in datasets.items():
        cols = np.arange(l, d['X'].shape[1], MAX_ORDER)
        mean = d['X'][:, cols].mean(axis=0)
        std = d['X'][:, cols].std(axis=0, ddof=1)
        rel = np.where(np.abs(mean) > 1e-12, std / np.abs(mean), np.nan)
        ax.plot(d['energies'], rel, '-o', ms=3, lw=1, color=COLORS[key], label=LABELS[key])
    ax.set_xlabel('Energy (MeV)')
    ax.set_ylabel(f'rel std of $a_{{{l + 1}}}$')
    ax.set_title(f'l = {l + 1}')
    ax.set_xscale('log')
    ax.set_yscale('log')
axes[0].legend(fontsize=8)
fig.tight_layout()

## Section 4 — Cross-bin correlation structure

For each Legendre order, slice out the (E_k, E_k') correlation block and compare. Then plot correlation vs energy distance to see how the energy decay differs.

In [ ]:
def cross_bin_block(corr, l, max_order, n_energies):
    idx = np.arange(n_energies) * max_order + l
    return corr[np.ix_(idx, idx)]


n_energies = len(next(iter(datasets.values()))['energies'])

fig, axes = plt.subplots(MAX_ORDER, len(datasets),
                         figsize=(4 * len(datasets), 4 * MAX_ORDER), squeeze=False)
for l in range(MAX_ORDER):
    for col, (key, _) in enumerate(datasets.items()):
        ax = axes[l, col]
        block = cross_bin_block(corrs[key], l, MAX_ORDER, n_energies)
        im = ax.imshow(block, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        ax.set_title(f'{LABELS[key]} — l = {l + 1}', fontsize=10)
        ax.set_xlabel('E index')
        ax.set_ylabel('E index')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, MAX_ORDER, figsize=(4.5 * MAX_ORDER, 4), sharey=True)
if MAX_ORDER == 1:
    axes = [axes]

energies = next(iter(datasets.values()))['energies']
for l in range(MAX_ORDER):
    ax = axes[l]
    for key, _ in datasets.items():
        block = cross_bin_block(corrs[key], l, MAX_ORDER, n_energies)
        di, dj = np.meshgrid(energies, energies, indexing='ij')
        dE = np.abs(di - dj)
        mask = np.triu(np.ones_like(block, dtype=bool), k=1)
        ax.plot(dE[mask], block[mask], '.', ms=3, alpha=0.5,
                color=COLORS[key], label=LABELS[key])
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xscale('log')
    ax.set_xlabel('|E_i − E_j|  (MeV)')
    ax.set_title(f'l = {l + 1}')
    if l == 0:
        ax.set_ylabel('Pearson correlation')
axes[0].legend(fontsize=8)
fig.tight_layout()

## Section 5 — Information-loss summary

Compact table of how much the Gaussian step distorts the joint distribution.

In [ ]:
if 'gaussian' in datasets and 'full_correlations' in datasets:
    Xg = datasets['gaussian']['X']
    Xf = datasets['full_correlations']['X']
    cg = corrs['gaussian']
    cf = corrs['full_correlations']
    covg = np.cov(Xg, rowvar=False)
    covf = np.cov(Xf, rowvar=False)

    def frob_rel(A, B):
        denom = np.linalg.norm(B, ord='fro')
        return float(np.linalg.norm(A - B, ord='fro') / denom) if denom > 0 else float('nan')

    skew_g, exkurt_g, _ = stats_by_set['gaussian']
    skew_f, exkurt_f, _ = stats_by_set['full_correlations']

    rows = [
        ('Frobenius rel. corr difference',
         f'{frob_rel(cf, cg):.3f}'),
        ('Max entrywise |corr_full − corr_gauss|',
         f'{np.nanmax(np.abs(cf - cg)):.3f}'),
        ('Frobenius rel. cov difference',
         f'{frob_rel(covf, covg):.3f}'),
        ('Fraction params with |skew| > 0.3 (full)',
         f'{np.mean(np.abs(skew_f) > 0.3):.2%}'),
        ('Fraction params with |skew| > 0.3 (gauss)',
         f'{np.mean(np.abs(skew_g) > 0.3):.2%}'),
        ('Fraction params with |excess kurt| > 0.5 (full)',
         f'{np.mean(np.abs(exkurt_f) > 0.5):.2%}'),
        ('Fraction params with |excess kurt| > 0.5 (gauss)',
         f'{np.mean(np.abs(exkurt_g) > 0.5):.2%}'),
        ('Mean MI − MI_gauss gap (key pairs, full)',
         f'{mi_df.loc[mi_df.sample_set == LABELS["full_correlations"], "gap"].mean():.3f}'),
        ('Mean MI − MI_gauss gap (key pairs, gauss)',
         f'{mi_df.loc[mi_df.sample_set == LABELS["gaussian"], "gap"].mean():.3f}'),
    ]
    summary_df = pd.DataFrame(rows, columns=['Quantity', 'Value'])
    print(summary_df.to_string(index=False))
else:
    print('Need both parquets to produce a summary.')

### Verdict (fill in after running)

Read the summary table together with the Q-Q plots, the difference heatmap, and the correlation-vs-distance scatter:

- If the relative Frobenius differences are small (say, < 0.05) and the marginal skew/kurtosis fractions are similar between the two sets, the Gaussian Cholesky step is not costing you anything material and the existing Pipeline A samples are fine to propagate as-is.
- If the difference heatmap shows structured residuals (especially long-range or off-diagonal), or the correlation-vs-distance plot disagrees noticeably between the two sets, the Pearson approximation is dropping real cross-bin structure. In that case, propagate the `full_correlations` samples directly (e.g. by re-running the pipeline with `KW_MC_TWO_PASS = False`, which writes those samples straight into Pipeline A's ENDF files).
- If the marginal skew/kurtosis fractions are large for the full set, the joint distribution has tails the Gaussian cannot represent and any covariance-only downstream propagation will under- or over-cover specific quantiles.